In [1]:
!poetry install -q

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import datetime
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# OpenMP 다중 로드 허용 및 스레드 경쟁 방지 환경 변수 (최상단 주입 필수)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

In [3]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [4]:
# ==============================================================================
# [셀 0] 주피터 분석 환경 전용 MinIO 및 MLflow 환경 변수 세팅
# ==============================================================================
import os
import warnings
import logging
from dotenv import find_dotenv, load_dotenv
from mlflow.tracking import MlflowClient
from pandas.errors import PerformanceWarning
from src.model.tracker.mlflow_tracker import MLflowTracker

# 1. 루트 디렉터리의 .env 파일 자동 탐색 및 로드
load_dotenv(find_dotenv())

# 2. MinIO S3 및 AWS 자격 증명 환경 변수 세팅
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION")
os.environ["LOCAL_S3_ENDPOINT"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")

# 3. MLflow Tracking & S3 아티팩트 스토어 연동 설정
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["MLFLOW_S3_IGNORE_TLS"] = "true"
# MLflow 공식 표준 출력 URL 링크 차단 환경변수 (소프트웨어 공식 플래그)
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"

# 4. 경고 필터링
warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# MLFlow 활성화
EXPERIMENT_NAME: str = "Champion_Dataset_Selection"
tracker = MLflowTracker(experiment_name=EXPERIMENT_NAME)

/Users/junsu/code/Project/AssetMind/apps/data-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🎯 [MLflow Tracker Active] Experiment: 'Champion_Dataset_Selection' (ID: 2)


In [5]:
# ==============================================================================
# [셀 1] 골드 레이어 18종 파생 데이터셋 고속 로드 및 실시간 스트리밍 관제
# ==============================================================================
from notebooks.utils.get_best_dataset.loader import GoldDatasetLoader

# ------------------------------------------------------------------------------
# Step 1. GoldDatasetLoader 인스턴스화 및 18종 골드 데이터셋 일괄 수집/정제
# ------------------------------------------------------------------------------
gold_dataset_loader = GoldDatasetLoader()
gold_dataset_repository, ingestion_summary_dataframe = gold_dataset_loader.load_all_datasets()

# ------------------------------------------------------------------------------
# Step 2. 전체 수집 정산 대시보드 사출
# ------------------------------------------------------------------------------
display(ingestion_summary_dataframe)


 📊 [Total Dataset Ingestion Summary Dashboard] Executed in 2027.25s


,Dataset Bucket ID,Total Rows,Total Features,Start Date,End Date,Load Time
0,bucket_impute_locf_detect_iqr_refine_clipping,3879,1456,2016-01-01,2026-08-25,120.67s
1,bucket_impute_locf_detect_iqr_refine_masking,3879,1456,2016-01-01,2026-08-25,110.59s
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,3879,1456,2016-01-01,2026-08-25,111.98s
3,bucket_impute_locf_detect_isolation_forest_refine_masking,3879,1456,2016-01-01,2026-08-25,115.81s
4,bucket_impute_locf_detect_zscore_refine_clipping,3879,1456,2016-01-01,2026-08-25,115.81s
5,bucket_impute_locf_detect_zscore_refine_masking,3879,1456,2016-01-01,2026-08-25,113.56s
6,bucket_impute_log_return_detect_iqr_refine_clipping,3879,1456,2016-01-01,2026-08-25,110.38s
7,bucket_impute_log_return_detect_iqr_refine_masking,3879,1456,2016-01-01,2026-08-25,112.14s
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,3879,1456,2016-01-01,2026-08-25,114.11s
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,3879,1456,2016-01-01,2026-08-25,110.54s


In [6]:
# [임시 테스트용] 데이터 슬라이싱 (추후 전체 백필 완료 시 주석 처리)
TEST_CUTOFF_DATE = "2026-08-21"
gold_dataset_repository = {
    bucket_id: df.loc[:TEST_CUTOFF_DATE].copy()
    for bucket_id, df in gold_dataset_repository.items()
}
print(f"[Test Mode Active] 18종 데이터셋을 {TEST_CUTOFF_DATE} 기준으로 슬라이싱 완료")

[Test Mode Active] 18종 데이터셋을 2026-08-21 기준으로 슬라이싱 완료


In [7]:
# ==============================================================================
# [셀 2] 18종 데이터셋 대상 파생 피처 생성 및 시계열 타겟 변수(T+20) 일괄 사출
# ==============================================================================
from src.feature.feature_service import FeatureService
from notebooks.utils.get_best_dataset.feature import (
    generate_domain_features,
    report_domain_groups
)

# ------------------------------------------------------------------------------
# Step 1. 18종 골드 데이터셋 일괄 파생 피처 엔지니어링 집행
# ------------------------------------------------------------------------------
feature_service = FeatureService()
engineered_dataset_repository, feature_summary_dataframe = generate_domain_features(
    datasets=gold_dataset_repository,
    feature_service=feature_service
)
display(feature_summary_dataframe)

# ------------------------------------------------------------------------------
# Step 2. 6대 도메인 피처 그룹 규격 및 사출 무결성 정밀 감사표 (Audit Report)
# ------------------------------------------------------------------------------
sample_bucket_id = next(iter(gold_dataset_repository.keys()))
domain_group_report = report_domain_groups(
    raw_sample=gold_dataset_repository[sample_bucket_id],
    engineered_sample=engineered_dataset_repository[sample_bucket_id]
)

print("\n" + "=" * 122)
print("[Feature Engineering Domain Group Specification Report]")
print("=" * 122)
display(domain_group_report)

 🚀 [Batch Feature Engineering Launch] Target Datasets: 18 Sets


Engineering Features: 100%|██████████| 18/18 [00:15<00:00,  1.14dataset/s, Processing: bucket_impute_moving_average_detect...]


 📊 [Feature Engineering Report] Completed in 15.85s


,Dataset Bucket ID,Input Shape,Output Shape,Generated Features,Target Status,Elapsed Time
0,bucket_impute_locf_detect_iqr_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.997s
1,bucket_impute_locf_detect_iqr_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.826s
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.863s
3,bucket_impute_locf_detect_isolation_forest_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.832s
4,bucket_impute_locf_detect_zscore_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.935s
5,bucket_impute_locf_detect_zscore_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.862s
6,bucket_impute_log_return_detect_iqr_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.880s
7,bucket_impute_log_return_detect_iqr_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.910s
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.914s
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.847s



[Feature Engineering Domain Group Specification Report]


,피처 그룹 (Feature Group),활성화 상태,표준 팩터 규격,총 사출 피처 수,표준 파생 팩터 목록
0,Target Feature,ENABLED,1 / 1 종,1 개,target_return_20d
1,Trend & Momentum (Multi-Asset),ENABLED,7 / 7 종,959 개,"return_lag_5d, return_lag_20d, return_lag_60d, return_lag_120d, ma_ratio_5_20, ma_ratio_20_60, risk_adjusted_return_20d"
2,Volatility & Risk (Multi-Asset),ENABLED,7 / 7 종,959 개,"volatility_20d, volatility_60d, vol_regime_ratio, rolling_skew_20d, rolling_kurt_20d, price_position_20d, norm_atr_20d"
3,Macro & Cross-Asset,DISABLED,0 / 4 종,0 개,-
4,Derivatives & Volume,ENABLED,3 / 4 종,3 개,"proxy_basis_rate, futures_intraday_range, volume_anomaly_20d"
5,Calendar & Seasonality,ENABLED,4 / 4 종,4 개,"month_sin, month_cos, is_month_end, is_quarter_end"


In [8]:
# ==============================================================================
# [셀 3] 18종 데이터셋 대상 Data Leakage 방지 시계열 일괄 분할 (Train / Test with Purged Gap)
# ==============================================================================
from src.model.dataset.splitter import DatasetSplitter
from notebooks.utils.get_best_dataset.splitter import batch_split_datasets, report_feature_leakage

# ------------------------------------------------------------------------------
# Step 1. DatasetSplitter 초기화 및 18종 데이터셋 일괄 시계열 분할
# ------------------------------------------------------------------------------
raw_columns_to_exclude = list(next(iter(gold_dataset_repository.values())).columns)

dataset_splitter = DatasetSplitter(
    split_ratios=(0.8, 0.2),
    forecast_horizon=20,
    exclude_features=raw_columns_to_exclude
)
split_dataset_repository = batch_split_datasets(
    datasets=engineered_dataset_repository,
    splitter=dataset_splitter
)

# ------------------------------------------------------------------------------
# Step 2. Data Leakage 전수 검증 및 분할 정산 리포트 사출
# ------------------------------------------------------------------------------
split_summary_report = report_feature_leakage(
    split_repository=split_dataset_repository,
    raw_columns=raw_columns_to_exclude
)
display(split_summary_report)

# ------------------------------------------------------------------------------
# Step 3. 대표 데이터셋 시계열 구간 및 Purged Gap 무결성 상세 리포트 사출
# ------------------------------------------------------------------------------
sample_bucket_id = next(iter(split_dataset_repository.keys()))
display(dataset_splitter.summarize(split_datasets=split_dataset_repository[sample_bucket_id]))

 🚀 [Batch Time-Series Splitting Launch] Mode: Train/Test (80:20) | Gap: 20d | Target: 18 Sets


,Dataset Bucket ID,Target Exclusion,X_train Shape,y_train Shape,purged_gap Shape,X_test Shape,y_test Shape,X_inference Shape
0,bucket_impute_locf_detect_iqr_refine_clipping,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
1,bucket_impute_locf_detect_iqr_refine_masking,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
3,bucket_impute_locf_detect_isolation_forest_refine_masking,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
4,bucket_impute_locf_detect_zscore_refine_clipping,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
5,bucket_impute_locf_detect_zscore_refine_masking,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
6,bucket_impute_log_return_detect_iqr_refine_clipping,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"
7,bucket_impute_log_return_detect_iqr_refine_masking,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"


,Partition,Date Range,Shape,Role
0,X_train,2016-01-01 ~ 2024-06-03,"(3,068, 1,925)",모델 가중치 학습용 피처 세트
1,y_train,2016-01-01 ~ 2024-06-03,"3,068",모델 가중치 학습용 타겟 벡터
2,purged_gap,2024-06-04 ~ 2024-06-23,"(20, 1,925)",데이터 누수 방지용 삭제 구간 (Purged Gap)
3,X_test,2024-06-24 ~ 2026-08-01,"(767, 1,925)",최종 검증(Out-of-Sample) 피처 세트
4,y_test,2024-06-24 ~ 2026-08-01,767,최종 검증(Out-of-Sample) 타겟 벡터
5,X_inference,2026-08-02 ~ 2026-08-21,"(20, 1,925)",실시간 추론 및 페이퍼 트레이딩 피처 세트 (y 결손)


In [9]:
# ==============================================================================
# [셀 4] 18종 데이터셋 피처 셀렉션 (Noise & Collinearity Filter ➔ Robust Scaling ➔ 2-Pillar Selection)
# ==============================================================================
import pandas as pd
from notebooks.utils.get_best_dataset.selector import batch_select_features

# ------------------------------------------------------------------------------
# Step 1. 피처 엔지니어링 후행 전처리 하이퍼파라미터 상수 정의
# ------------------------------------------------------------------------------
MAX_MISSING_RATIO: float = 0.2
MAX_ZERO_RATIO: float = 0.8
PEARSON_THRESHOLD: float = 0.85
DISTANCE_THRESHOLD: float = 0.40
TARGET_CUMULATIVE_THRESHOLD: float = 0.95
MIN_FEATURES_BOUND: int = 10
MAX_FEATURES_BOUND: int = 100

# ------------------------------------------------------------------------------
# Step 2. 18종 데이터셋 일괄 5단계 후행 전처리 및 동적 피처 선별 집행
# ------------------------------------------------------------------------------
model_ready_repository, selection_summary_report, representative_audit_report = batch_select_features(
    split_repository=split_dataset_repository,
    max_missing_ratio=MAX_MISSING_RATIO,
    max_zero_ratio=MAX_ZERO_RATIO,
    pearson_threshold=PEARSON_THRESHOLD,
    distance_threshold=DISTANCE_THRESHOLD,
    cumulative_threshold=TARGET_CUMULATIVE_THRESHOLD,
    min_features=MIN_FEATURES_BOUND,
    max_features=MAX_FEATURES_BOUND
)

# ------------------------------------------------------------------------------
# Step 3. 18개 데이터셋 피처 셀렉션 종합 정산 대시보드 사출
# ------------------------------------------------------------------------------
display(selection_summary_report)

# ------------------------------------------------------------------------------
# Step 4. 대표 데이터셋 5단계 후행 전처리 세부 감사표 (Audit Table) 사출
# ------------------------------------------------------------------------------
sample_dataset_id = next(iter(split_dataset_repository.keys()))
print(f"\n[Representative Dataset Feature Selection Audit: '{sample_dataset_id}']")
display(representative_audit_report)

 🚀 [Batch Feature Selection Launch] Target Datasets: 18 Sets | 2-Pillar Hybrid + Robust Scaler


🔍 [2-Pillar Selection]: 100%|██████████| 18/18 [11:49<00:00, 39.40s/dataset, Done: bucket_impute_moving_aver... (19.1s, Feats: 12)]

 ✅ [Batch Feature Selection Completed] Total Elapsed Time: 709.13s


,Dataset Bucket ID,Initial Features,After Noise Filter,After Collinear Filter,Selected Features,Cumulative Coverage,Elapsed Time
0,bucket_impute_locf_detect_iqr_refine_clipping,1925,1883,608,59,95.0%,40.56s
1,bucket_impute_locf_detect_iqr_refine_masking,1925,1863,798,57,95.1%,48.81s
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,1925,1883,608,59,95.0%,39.30s
3,bucket_impute_locf_detect_isolation_forest_refine_masking,1925,1863,490,10,95.6%,18.51s
4,bucket_impute_locf_detect_zscore_refine_clipping,1925,1883,608,59,95.0%,39.21s
5,bucket_impute_locf_detect_zscore_refine_masking,1925,1863,493,15,95.2%,18.50s
6,bucket_impute_log_return_detect_iqr_refine_clipping,1925,1882,578,84,95.1%,47.08s
7,bucket_impute_log_return_detect_iqr_refine_masking,1925,1862,849,79,95.1%,62.85s
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,1925,1882,578,84,95.1%,47.20s
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,1925,1867,500,30,95.1%,18.61s



[Representative Dataset Feature Selection Audit: 'bucket_impute_locf_detect_iqr_refine_clipping']


,잔여 피처 수 (Features),변동 내역 (Changes),적용 기준 (Fit Strategy)
단계 (Pipeline Step),,,
1. Noise Filter (Constant & Missing & Zero),"1,883","-42 Features (Constant: 0, Missing: 15, Zero: 27)","상수 및 결측률(> 20%), 0값(≥ 80%)"
2. Pearson Collinearity,979,-904 Features,피어슨 선형 상관계수(|r| ≥ 0.85)
3. Hierarchical Feature Clustering (HFC),608,-371 Features,상관거리 계층 군집화 (Distance ≤ 0.4)
4. Robust Feature Scaling,608,No Dimension Change (All Partitions Normalized),학습 데이터 중앙값 및 IQR 통계량 기준
5. Dynamic 2-Pillar Selection (Top 59),59,Selected 59 Features,ElasticNet + LightGBM (Target: 95%)


In [10]:
# ==============================================================================
# [셀 5] 18종 데이터셋 대상 Expanding Walk-Forward CV 시계열 분할 무결성 감사
# ==============================================================================
from src.model.dataset.walk_forward_splitter import WalkForwardSplitter

# ------------------------------------------------------------------------------
# Step 1. WalkForwardSplitter 인스턴스화 (5-Fold Expanding Window & 20d Purged Gap)
# ------------------------------------------------------------------------------
walk_forward_splitter = WalkForwardSplitter(
    n_splits=5,
    forecast_horizon=20,
    min_train_ratio=0.5
)

# ------------------------------------------------------------------------------
# Step 2. 대표 데이터셋 X_train(80% 학습 영역) 기준 시계열 CV 분할 감사표 사출
# ------------------------------------------------------------------------------
sample_bucket_id = next(iter(model_ready_repository.keys()))
sample_X_train = model_ready_repository[sample_bucket_id]["X_train"]
sample_y_train = model_ready_repository[sample_bucket_id]["y_train"]

walk_forward_audit_df = walk_forward_splitter.summarize(
    X=sample_X_train,
    y=sample_y_train
)

# ------------------------------------------------------------------------------
# Step 3. 시계열 분할 감사 배너 및 Fold별 구간 정밀 감사 대시보드 사출
# ------------------------------------------------------------------------------
print("=" * 122)
print(f" 🔬 [Expanding-Window Walk-Forward CV Audit Dashboard] Target: '{sample_bucket_id}' (Total {len(walk_forward_audit_df)} Folds)")
print("=" * 122)
display(walk_forward_audit_df)

 🔬 [Expanding-Window Walk-Forward CV Audit Dashboard] Target: 'bucket_impute_locf_detect_iqr_refine_clipping' (Total 5 Folds)


,Train 기간 (Expanding),Train 규격,Purged Gap (20d),Gap 규격,Validation 기간,Val 규격,검증 비고
Fold,,,,,,,
Fold 1,2016-01-01 ~ 2020-03-02,"1,518 Rows",2020-03-03 ~ 2020-03-22,20 Rows,2020-03-23 ~ 2021-01-23,306 Rows,Train 49.5% ➔ Val 10.0%
Fold 2,2016-01-01 ~ 2021-01-03,"1,824 Rows",2021-01-04 ~ 2021-01-23,20 Rows,2021-01-24 ~ 2021-11-25,306 Rows,Train 59.5% ➔ Val 10.0%
Fold 3,2016-01-01 ~ 2021-11-05,"2,130 Rows",2021-11-06 ~ 2021-11-25,20 Rows,2021-11-26 ~ 2022-09-28,306 Rows,Train 69.4% ➔ Val 10.0%
Fold 4,2016-01-01 ~ 2022-09-08,"2,436 Rows",2022-09-09 ~ 2022-09-28,20 Rows,2022-09-29 ~ 2023-08-01,306 Rows,Train 79.4% ➔ Val 10.0%
Fold 5,2016-01-01 ~ 2023-07-12,"2,742 Rows",2023-07-13 ~ 2023-08-01,20 Rows,2023-08-02 ~ 2024-06-03,306 Rows,Train 89.4% ➔ Val 10.0%


In [11]:
# ==============================================================================
# [셀 6] 18종 데이터셋 × 5-Fold Walk-Forward CV 3대 모델 스크리닝 및 챔피언 확정
# ==============================================================================
from notebooks.utils.get_best_dataset.screener import run_batch_cv_screening, save_champion_dataset

# ------------------------------------------------------------------------------
# Step 1. 18종 데이터셋 × 5-Fold Expanding CV × 3대 베이스라인 모델 일괄 스크리닝 (270 Evals)
# ------------------------------------------------------------------------------
raw_ranking_report, champion_dataset_id, champion_meta, display_ranking_report = run_batch_cv_screening(
    model_ready_repository=model_ready_repository,
    n_splits=5,
    forecast_horizon=20,
    min_train_ratio=0.5
)

# ------------------------------------------------------------------------------
# Step 2. 완성형 1위 Champion Dataset 파티션 직렬화 저장 및 무결성 검증
# ------------------------------------------------------------------------------
artifact_meta = save_champion_dataset(
    model_ready_repository=model_ready_repository,
    champion_dataset_id=champion_dataset_id,
    output_artifact_path="champion_dataset.pkl"
)

# ------------------------------------------------------------------------------
# Step 3. 챔피언 확정 배너 및 18종 CV 스크리닝 종합 정산 대시보드 사출
# ------------------------------------------------------------------------------
print("\n" + "=" * 122)
print(f" 🏆 [Champion Dataset Selected via 5-Fold Walk-Forward CV] '{champion_dataset_id}' 확정!")
print(f" 📌 5-Fold CV 성능: Avg MDA {champion_meta['avg_mda'] * 100:.2f}% | Avg RMSE {champion_meta['avg_rmse']:.6f} | Selected Features: {champion_meta['features']}개")
print(f" 💾 완성형 데이터셋 파티션 저장 완료: '{artifact_meta['artifact_path']}' (크기: {artifact_meta['file_size_kb']:.2f} KB | Train: {artifact_meta['train_shape']} | Test: {artifact_meta['test_shape']})")
print(f" ⏱️ 총 소요 시간: {champion_meta['total_elapsed']:.2f}s (총 270회 모델 검증 완주)")
print("=" * 122)

display(display_ranking_report)

 🚀 [Batch Walk-Forward CV Screening Launch] Target: 18 Sets × 5 Folds × 3 Models (270 Evaluations)


🔍 [5-Fold CV Screening]: 100%|██████████| 18/18 [00:15<00:00,  1.16set/s]

 ✅ [Screening Completed] Total Elapsed Time: 15.47s

 🏆 [Champion Dataset Selected via 5-Fold Walk-Forward CV] 'bucket_impute_log_return_detect_zscore_refine_masking' 확정!
 📌 5-Fold CV 성능: Avg MDA 73.29% | Avg RMSE 0.430884 | Selected Features: 25개
 💾 완성형 데이터셋 파티션 저장 완료: 'champion_dataset.pkl' (크기: 1280.90 KB | Train: (3047, 25) | Test: (762, 25))
 ⏱️ 총 소요 시간: 15.47s (총 270회 모델 검증 완주)


,Selected Features,Avg MDA,Avg RMSE,Avg MAE,Best Model,Best Model MDA,Best Model RMSE,Dataset Bucket ID,Elapsed Time
Rank,,,,,,,,,
1,25,73.29%,0.430884,0.302675,ElasticNet,73.73%,0.442633,bucket_impute_log_return_detect_zscore_refine_masking,0.68s
2,30,72.39%,0.472191,0.338768,ElasticNet,73.66%,0.442608,bucket_impute_log_return_detect_isolation_forest_refine_masking,0.74s
3,10,71.41%,0.340275,0.245275,XGBoostRegressor,71.80%,0.306557,bucket_impute_locf_detect_isolation_forest_refine_masking,0.57s
4,15,71.06%,0.341208,0.245396,XGBoostRegressor,71.28%,0.306219,bucket_impute_locf_detect_zscore_refine_masking,0.58s
5,12,70.91%,0.332495,0.237625,XGBoostRegressor,75.93%,0.298239,bucket_impute_moving_average_detect_zscore_refine_masking,0.64s
6,14,70.73%,0.330001,0.235782,XGBoostRegressor,75.93%,0.295519,bucket_impute_moving_average_detect_isolation_forest_refine_masking,0.61s
7,84,68.69%,0.203550,0.106603,RandomForestRegressor,70.23%,0.181850,bucket_impute_log_return_detect_iqr_refine_clipping,1.26s
8,84,68.69%,0.203550,0.106603,RandomForestRegressor,70.23%,0.181850,bucket_impute_log_return_detect_isolation_forest_refine_clipping,1.30s
9,84,68.69%,0.203550,0.106603,RandomForestRegressor,70.23%,0.181850,bucket_impute_log_return_detect_zscore_refine_clipping,1.26s
